# VNFood Vision — Training Pipeline trên Google Colab

**Hướng dẫn:**
1. Vào `Runtime` → `Change runtime type` → Chọn **T4 GPU** → Save
2. Chạy từng ô theo thứ tự từ trên xuống dưới
3. Chỉ cần sửa ô **CẤU HÌNH** (ô số 2) cho đúng đường dẫn Google Drive của bạn


## ⚙️ BƯỚC 0 — CẤU HÌNH (SỬA Ô NÀY TRƯỚC KHI CHẠY)

In [ ]:
# Đường dẫn đến thư mục dự án trên Google Drive của bạn
# (Thư mục chứa file README.md, Task.md, src/, configs/...)
PROJECT_DIR = "/content/drive/MyDrive/vnfood_vision"

# Đường dẫn đến thư mục ảnh đã xử lý
DATA_DIR = "/content/drive/MyDrive/VietFood-Project/data/processed"


import os
os.environ['VNFOOD_DATA_DIR'] = DATA_DIR
print(f"✅ Project dir : {PROJECT_DIR}")
print(f"✅ Data dir    : {DATA_DIR}")

✅ Project dir : /content/drive/MyDrive/vnfood_vision
✅ Data dir    : /content/drive/MyDrive/VietFood-Project/data/processed


## BƯỚC 1 — Mount Google Drive & Kiểm tra GPU

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Kiểm tra GPU
import torch
print(f"\n🖥️  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU — Vào Runtime > Change runtime type > T4 GPU'}")
print(f"   CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

Mounted at /content/drive

🖥️  GPU: Tesla T4
   CUDA available: True
   VRAM: 14.6 GB


## BƯỚC 2 — Cài đặt thư viện

In [ ]:
# Cài các thư viện cần thiết (Colab đã có torch/torchvision rồi)
!pip install -q pyyaml tqdm scikit-learn pillow
print("✅ Cài đặt xong!")

✅ Cài đặt xong!


## BƯỚC 3 — Copy code vào Colab & Kiểm tra cấu trúc

In [ ]:
import shutil, os

# Copy toàn bộ thư mục dự án từ Drive vào /content/vnfood_vision
# (Đọc file từ /content/ nhanh hơn Drive ~10 lần)
LOCAL_PROJECT = "/content/vnfood_vision"

if os.path.exists(LOCAL_PROJECT):
    shutil.rmtree(LOCAL_PROJECT)

shutil.copytree(PROJECT_DIR, LOCAL_PROJECT)
os.chdir(LOCAL_PROJECT)
print(f"✅ Code đã copy vào: {LOCAL_PROJECT}")
print(f"📁 Thư mục hiện tại: {os.getcwd()}")

# Kiểm tra cấu trúc
!ls -la

✅ Code đã copy vào: /content/vnfood_vision
📁 Thư mục hiện tại: /content/vnfood_vision
total 72
drwx------ 6 root root  4096 Jun  8 16:54 .
drwxr-xr-x 1 root root  4096 Jun  9 04:35 ..
drwx------ 2 root root  4096 Jun  8 16:46 checkpoints
drwx------ 2 root root  4096 Jun  8 16:04 configs
-rw------- 1 root root  2466 Jun  8 16:04 .gitignore
drwx------ 2 root root  4096 Jun  8 16:04 notebooks
-rw------- 1 root root  2399 Jun  8 16:04 PROJECT_STATE.md
-rw------- 1 root root  4434 Jun  8 16:04 README.md
-rw------- 1 root root   313 Jun  8 16:04 requirements.txt
drwx------ 2 root root  4096 Jun  8 16:04 src
-rw------- 1 root root 12366 Jun  8 16:04 Task_2_3.md
-rw------- 1 root root 11425 Jun  8 16:04 Task.md


In [ ]:
# Kiểm tra thư mục data (QUAN TRỌNG)
import os
from pathlib import Path

data_path = Path(DATA_DIR)
if data_path.exists():
    # Đếm số ảnh
    all_images = list(data_path.rglob('*.jpg')) + list(data_path.rglob('*.jpeg')) + list(data_path.rglob('*.png'))
    classes = [d.name for d in data_path.rglob('*') if d.is_dir() and not any(d.name.startswith(x) for x in ['.'])]
    print(f"Tìm thấy thư mục data!")
    print(f"   Tổng số ảnh: {len(all_images):,}")
    print(f"   Số thư mục con: {len(classes)}")
    print(f"   Ví dụ 5 thư mục đầu: {[d.name for d in list(data_path.iterdir())[:5]]}")
else:
    print(f"KHÔNG TÌM THẤY: {DATA_DIR}")
    print("   Kiểm tra lại đường dẫn DATA_DIR ở ô CẤU HÌNH")

Tìm thấy thư mục data!
   Tổng số ảnh: 27,868
   Số thư mục con: 69
   Ví dụ 5 thư mục đầu: ['bac', 'trung', 'chung', 'nam']


## BƯỚC 4 — TRAIN MÔ HÌNH

In [ ]:
# Ô này chạy ~10-15 phút nhưng giúp training nhanh gấp 10-20 lần!
import shutil
print("Đang copy ảnh từ Drive về local... (chờ 10-15 phút)")
shutil.copytree(
    "/content/drive/MyDrive/VietFood-Project/data/processed",  # ← Drive
    "/content/data/processed"                                   # ← Local Colab
)
print("✅ Copy xong! Giờ train sẽ nhanh gấp 10-20 lần!")


Đang copy ảnh từ Drive về local... (chờ 10-15 phút)
✅ Copy xong! Giờ train sẽ nhanh gấp 10-20 lần!


In [ ]:
# Chạy training!
# - Tự động dùng GPU T4
# - Tự động lưu model tốt nhất vào checkpoints/best_model.pth
# - Tự động dừng nếu không cải thiện sau 10 epoch (Early Stopping)

!python src/train.py \
    --config configs/config.yaml \
    --backbone efficientnet_b3

  Data dir : /content/drive/MyDrive/VietFood-Project/data/processed

  VNFood Vision — Training
  Device : cuda
  Backbone: efficientnet_b3

  Dataset: 27,868 ảnh | 43 class
  Train: 19,508 | Val: 4,180 | Test: 4,180

[Model] Class weights saved → configs/class_weights.json
  ✅ WeightedRandomSampler: Bật — cân bằng 43 class trong mỗi batch
[Model] Backbone frozen. Chỉ train classifier.

🔄 Phát hiện checkpoint cũ! Đang resume từ: checkpoints/last_checkpoint.pth
   ✅ Resume từ Epoch 10 | Best acc: 70.50%
   Tiếp tục từ Epoch 11...

  💾 Drive sync: Bật — sẽ tự động lưu về /content/drive/MyDrive/vnfood_vision/checkpoints sau mỗi epoch
Epoch 011/20 | LR: 8.85e-04 | Train Loss: 0.3572 Acc: 37.32% | Val Loss: 0.2784 | Val Top-1: 70.14% Top-5: 90.98% | 466s
  💾 Đã sync → Drive (11/20)
Epoch 012/20 | LR: 8.65e-04 | Train Loss: 0.3459 Acc: 36.41% | Val Loss: 0.2670 | Val Top-1: 72.11% Top-5: 91.51% | 452s
  ✅ Best model saved! Val Top-1: 72.11%
  💾 Đã sync → Drive (12/20)
Epoch 013/20 | LR: 8.42

In [ ]:
# [TÙY CHỌN] So sánh với backbone ResNet50
# Chạy ô này sau khi training EfficientNet xong để so sánh

!python src/train.py \
    --config configs/config.yaml \
    --backbone mobilenet_v3_large

  Data dir : /content/drive/MyDrive/VietFood-Project/data/processed

  VNFood Vision — Training
  Device : cuda
  Backbone: mobilenet_v3_large

  Dataset: 27,868 ảnh | 43 class
  Train: 19,508 | Val: 4,180 | Test: 4,180

[Model] Class weights saved → configs/class_weights.json
  ✅ WeightedRandomSampler: Bật — cân bằng 43 class trong mỗi batch
[Model] Backbone frozen. Chỉ train classifier.

🔄 Phát hiện checkpoint cũ! Đang resume từ: checkpoints/mobilenet_v3_large/last_checkpoint.pth
   ✅ Resume từ Epoch 2 | Best acc: 0.00%
   Tiếp tục từ Epoch 3...

  💾 Drive sync: Bật — sẽ tự động lưu về /content/drive/MyDrive/vnfood_vision/checkpoints/mobilenet_v3_large sau mỗi epoch
[Model] Backbone unfrozen. Fine-tuning toàn bộ mạng.

[Epoch 3] Unfreeze backbone! Tiếp tục fine-tune toàn bộ.
  📉 Đã giảm Learning Rate 10 lần để chống sốc (Catastrophic Forgetting)!

Epoch 003/20 | LR: 9.91e-04 | Train Loss: 0.2020 Acc: 81.72% | Val Loss: 0.3524 | Val Top-1: 62.34% Top-5: 87.20% | 418s
  ✅ Best model sa

## BƯỚC 5 — ACTIVE LEARNING (AI tự lọc ảnh khả nghi)

In [ ]:
# Chạy Active Learning Scanner
# AI sẽ tự quét 27k ảnh và tự quyết định cần review bao nhiêu ảnh

!python src/active_learning.py \
    --data_dir "{DATA_DIR}" \
    --model_path "checkpoints/best_model.pth"

In [ ]:
# Copy kết quả Active Learning về Google Drive để dùng với Label Studio
import shutil
from pathlib import Path

src = Path("/content/vnfood_vision/outputs/active_learning")
dst = Path(PROJECT_DIR) / "outputs" / "active_learning"
dst.mkdir(parents=True, exist_ok=True)

if src.exists():
    for f in src.iterdir():
        shutil.copy(f, dst / f.name)
    print(f"✅ Kết quả đã copy về Drive: {dst}")
    print(f"   File để import Label Studio: {dst / 'label_studio_import.json'}")
else:
    print("⚠️  Chưa có kết quả. Chạy ô Active Learning trước.")

## BƯỚC 6 — TEST NHẬN DIỆN 1 ẢNH

In [ ]:
# Test nhận diện một ảnh bất kỳ
# Thay đường dẫn bên dưới bằng ảnh bạn muốn test

TEST_IMAGE = "/content/drive/MyDrive/VietFood-Project/banh_cuon.jpg"

!python src/inference.py \
    --image "{TEST_IMAGE}" \
    --model_path "checkpoints/best_model.pth" \
    --tta  # Bật Test Time Augmentation để chính xác hơn

## BƯỚC 7 — LƯU MODEL VỀ GOOGLE DRIVE

In [ ]:
# Copy toàn bộ checkpoints về Drive để không mất khi Colab tắt
import shutil
from pathlib import Path

src = Path("/content/vnfood_vision/checkpoints")
dst = Path(PROJECT_DIR) / "checkpoints"
dst.mkdir(parents=True, exist_ok=True)

if src.exists():
    for f in src.iterdir():
        shutil.copy(f, dst / f.name)
        print(f"✅ Đã lưu: {f.name} ({f.stat().st_size/1024/1024:.1f} MB)")
    print(f"\n📁 Model đã lưu tại Drive: {dst}")
else:
    print("⚠️  Chưa có checkpoint. Hãy chạy Training trước.")